In [ ]:
# ==========================================
# AMAZON PRODUCT RATING PREDICTION - ALL IN ONE
# File: amazon_rating_prediction.ipynb
# Dataset: amazon.csv
# Target: rating
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------------------
# 1) Load dataset
# ----------------------------
df = pd.read_csv("/content/amazon.csv (1).zip")
print("Shape:", df.shape)
print(df.head())

# ----------------------------
# 2) Basic cleaning helpers
# ----------------------------
def clean_price(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r"[^\d.]", "", x)   # remove ₹, commas, spaces, etc.
    try:
        return float(x) if x != "" else np.nan
    except:
        return np.nan

def clean_count(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r"[^\d]", "", x)
    try:
        return float(x) if x != "" else np.nan
    except:
        return np.nan

def clean_rating(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r"[^\d.]", "", x)
    try:
        return float(x) if x != "" else np.nan
    except:
        return np.nan

# ----------------------------
# 3) Standardize column names
# ----------------------------
df.columns = [c.strip().lower() for c in df.columns]

# ----------------------------
# 4) Keep only useful columns if they exist
# ----------------------------
expected_cols = [
    "product_id", "product_name", "category", "discounted_price", "actual_price",
    "discount_percentage", "rating", "rating_count", "about_product",
    "review_title", "review_content"
]

for col in expected_cols:
    if col not in df.columns:
        df[col] = np.nan

# ----------------------------
# 5) Data cleaning
# ----------------------------
df = df.drop_duplicates().copy()

df["discounted_price"] = df["discounted_price"].apply(clean_price)
df["actual_price"] = df["actual_price"].apply(clean_price)
df["discount_percentage"] = df["discount_percentage"].apply(clean_count)
df["rating"] = df["rating"].apply(clean_rating)
df["rating_count"] = df["rating_count"].apply(clean_count)

# Fill text columns
text_cols = ["product_name", "category", "about_product", "review_title", "review_content"]
for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

# Remove rows without target
df = df.dropna(subset=["rating"]).copy()

print("\nCleaned Shape:", df.shape)
print(df.info())

# ----------------------------
# 6) Feature engineering
# ----------------------------
# Combine text into one column for TF-IDF
df["combined_text"] = (
    df["product_name"].fillna("") + " " +
    df["category"].fillna("") + " " +
    df["about_product"].fillna("") + " " +
    df["review_title"].fillna("") + " " +
    df["review_content"].fillna("")
).str.strip()

# Create a few simple numeric features
df["review_length"] = df["review_content"].apply(lambda x: len(str(x).split()))
df["about_length"] = df["about_product"].apply(lambda x: len(str(x).split()))
df["discount_ratio"] = np.where(
    (df["actual_price"].notna()) & (df["actual_price"] > 0) & (df["discounted_price"].notna()),
    df["discounted_price"] / df["actual_price"],
    np.nan
)

# ----------------------------
# 7) Quick EDA
# ----------------------------
plt.figure(figsize=(8, 4))
sns.histplot(df["rating"], bins=20, kde=True)
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
top_categories = df["category"].value_counts().head(10)
sns.barplot(x=top_categories.values, y=top_categories.index)
plt.title("Top 10 Categories")
plt.xlabel("Count")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["discount_percentage"].dropna(), bins=20, kde=True)
plt.title("Discount Percentage Distribution")
plt.xlabel("Discount Percentage")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["actual_price"].dropna(), bins=20, kde=True)
plt.title("Actual Price Distribution")
plt.xlabel("Actual Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["discounted_price"].dropna(), bins=20, kde=True)
plt.title("Discounted Price Distribution")
plt.xlabel("Discounted Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x="discount_percentage", y="rating")
plt.title("Discount Percentage vs Rating")
plt.tight_layout()
plt.show()

# Correlation heatmap for numeric columns
numeric_cols_for_corr = [
    "discounted_price", "actual_price", "discount_percentage",
    "rating", "rating_count", "review_length", "about_length", "discount_ratio"
]
corr = df[numeric_cols_for_corr].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# ----------------------------
# 8) Select features and target
# ----------------------------
feature_cols = [
    "combined_text",
    "category",
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating_count",
    "review_length",
    "about_length",
    "discount_ratio"
]

X = df[feature_cols].copy()
y = df["rating"].copy()

# ----------------------------
# 9) Train-test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------
# 10) Preprocessing
# ----------------------------
text_transformer = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english"))
])

numeric_features = [
    "discounted_price", "actual_price", "discount_percentage",
    "rating_count", "review_length", "about_length", "discount_ratio"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_features = ["category"]

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("text", text_transformer, "combined_text"),
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features),
    ],
    remainder="drop"
)

# ----------------------------
# 11) Model
# ----------------------------
model = Ridge(alpha=1.0, random_state=42)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# ----------------------------
# 12) Train
# ----------------------------
pipeline.fit(X_train, y_train)

# ----------------------------
# 13) Predict
# ----------------------------
y_pred = pipeline.predict(X_test)

# ----------------------------
# 14) Evaluation
# ----------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== MODEL PERFORMANCE =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ----------------------------
# 15) Actual vs Predicted plot
# ----------------------------
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Actual vs Predicted Ratings")
plt.tight_layout()
plt.show()

# ----------------------------
# 16) Residual plot
# ----------------------------
residuals = y_test - y_pred

plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=30, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residuals")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# ----------------------------
# 17) Show sample predictions
# ----------------------------
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
}).head(15)

print("\nSample Predictions:")
print(results)

# ----------------------------
# 18) Conclusion text
# ----------------------------
print("\n===== CONCLUSION =====")
print("This notebook cleaned Amazon product review data, performed EDA,")
print("engineered text and numeric features, trained a rating prediction model,")
print("and evaluated it using MAE, RMSE, and R2 score.")